# FoKL-to-Pyomo Example Syntax

## Setup

Import modules:

In [1]:
from FoKL import FoKLRoutines
import os
dir = os.path.abspath('')  # directory of notebook
# -----------------------------------------------------------------------
# UNCOMMENT IF USING LOCAL FOKL PACKAGE:
import sys
sys.path.append(os.path.join(dir, '..', '..'))  # package directory
from src.FoKL import FoKLRoutines
# -----------------------------------------------------------------------
import pyomo.environ as pyo
import pyomo.dae as dae
import numpy as np
import warnings

Example GP:

In [2]:
time = np.linspace(0, 99, 100)
noise = np.random.rand(time.size) - 0.5
position = np.sin(6 * np.pi * time / time[-1])
velocity = np.gradient(position)
acceleration = np.gradient(velocity)

GP = FoKLRoutines.FoKL(kernel=1, UserWarnings=False)
_ = GP.fit([acceleration, velocity], position + noise, clean=True)

[1, -146.5690488930369]
[2, -143.00052817007875]
[2, -143.00052817007875]
[3, -140.50395624706974]


## Embed GP in Pyomo

Arguments:

In [3]:
draws = 5
gp_name = 'GP name test'

mtx = GP.mtx
betas = GP.betas
minmax = GP.minmax
phis = GP.phis

t_span = [time[0], time[-1]]  # acts like 't_span' of 'solve_ivp'

m = pyo.ConcreteModel('Pyomo model test')
m.t = dae.ContinuousSet(bounds=t_span)

m.p = pyo.Var(m.t, initialize=0.0, domain=pyo.Reals)  # position
m.v = dae.DerivativeVar(m.p, wrt=m.t, initialize=0.0)  # velocity
m.a = dae.DerivativeVar(m.v, wrt=m.t, initialize=0.0)  # acceleration

xvars = [m.a, m.v]
yvar = m.p

FoKL-to-Pyomo:

In [4]:
m = GP.to_pyomo(xvars, yvar, m, 3)

Pyomo model with GP:

In [5]:
m.pprint()

1 Var Declarations
    p : Size=2, Index=t
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          0 :  None :   0.0 :  None : False : False :  Reals
         99 :  None :   0.0 :  None : False : False :  Reals

1 Block Declarations
    GP0 : Size=1, Index=None, Active=True
        4 Set Declarations
            attributes : Size=1, Index=None, Ordered=Insertion
                Key  : Dimen : Domain : Size : Members
                None :     1 :    Any :    2 : {0, 1}
            draws : Size=1, Index=None, Ordered=Insertion
                Key  : Dimen : Domain : Size : Members
                None :     1 :    Any :    3 : {0, 1, 2}
            orders : Size=1, Index=None, Ordered=Insertion
                Key  : Dimen : Domain : Size : Members
                None :     1 :    Any :    1 :    {1,}
            terms : Size=1, Index=None, Ordered=Insertion
                Key  : Dimen : Domain : Size : Members
                None :     1 :    Any :    2 : {0, 1}

    